# Metadata Curation with LLM Assistant

This notebook demonstrates how to use the MetadataAssistant to curate
table and column metadata with LLM-powered suggestions.

## Prerequisites

Set the `ANTHROPIC_API_KEY` environment variable for LLM suggestions.

In [ ]:
from analysis_helpers import MinioExplorer, display_table_preview
from metadata_assistant import (
    MetadataAssistant,
    TableMetadata,
    ColumnMetadata,
    CatalogClient,
)

## 1. Load a Document and Table

In [ ]:
explorer = MinioExplorer()

# Find available data
agencies = explorer.list_agencies("parsed-zone")
print("Agencies:", agencies)

if agencies:
    assets = explorer.list_assets(agencies[0], "parsed-zone")
    print(f"\nAssets for {agencies[0]}:", [a.asset for a in assets])

In [ ]:
# Load a document
if agencies and assets:
    doc = explorer.load_document(agencies[0], assets[0].asset)
    
    # List tables
    tables = explorer.list_tables(doc)
    for t in tables:
        print(f"Table {t['index']}: {t['title']} ({t['rows']} rows)")

In [ ]:
# Preview the first table
if 'doc' in dir() and tables:
    df = explorer.load_table(doc, table_index=0)
    display_table_preview(df, title=tables[0]['title'])

## 2. Create the Metadata Assistant

In [ ]:
# Create the assistant
assistant = MetadataAssistant()
print("Assistant created with model:", assistant.model)

## 3. Generate Metadata Suggestions

The assistant uses Claude to analyze the table and suggest metadata.

In [ ]:
# Generate suggestions for the first table
if 'doc' in dir() and tables:
    print("Generating metadata suggestions...")
    metadata = assistant.suggest_table_metadata(doc, table_index=0)
    
    print("\n=== Suggested Metadata ===")
    print(f"Display Name: {metadata.display_name}")
    print(f"Description: {metadata.description}")
    print(f"Data Domain: {metadata.data_domain}")
    print(f"\nColumns ({len(metadata.columns)}):")
    for col in metadata.columns:
        print(f"  - {col.column_name}:")
        print(f"      Display: {col.display_name}")
        print(f"      Type: {col.data_type}")
        if col.semantic_type:
            print(f"      Semantic: {col.semantic_type}")

## 4. Review and Modify Suggestions

In [ ]:
# Modify the suggested metadata as needed
if 'metadata' in dir():
    # Example: customize the display name
    # metadata.display_name = "My Custom Table Name"
    
    # Example: update a column description
    # for col in metadata.columns:
    #     if col.column_name == "some_column":
    #         col.description = "Updated description"
    
    print("Current metadata:")
    print(f"  Table ID: {metadata.table_id}")
    print(f"  Display Name: {metadata.display_name}")

In [ ]:
# Get refined suggestions for a specific column
if 'doc' in dir() and tables and metadata.columns:
    col_name = metadata.columns[0].column_name
    print(f"Getting refined suggestions for column: {col_name}")
    
    col_metadata = assistant.suggest_column_metadata(doc, table_index=0, column_name=col_name)
    
    print(f"\nColumn: {col_metadata.column_name}")
    print(f"Display Name: {col_metadata.display_name}")
    print(f"Description: {col_metadata.description}")
    print(f"Data Type: {col_metadata.data_type}")
    print(f"Semantic Type: {col_metadata.semantic_type}")
    print(f"Sample Values: {col_metadata.sample_values}")

## 5. Validate Metadata

In [ ]:
# Validate the metadata
if 'metadata' in dir():
    validation = assistant.validate_metadata(metadata)
    
    print(f"Valid: {validation.is_valid}")
    print(f"Completeness: {validation.completeness_score:.1%}")
    
    if validation.issues:
        print(f"\nIssues ({len(validation.issues)}):")
        for issue in validation.issues[:10]:  # Show first 10
            print(f"  [{issue.severity.upper()}] {issue.field}: {issue.message}")

In [ ]:
# Check coverage against actual data
if 'metadata' in dir() and 'df' in dir():
    coverage = assistant.check_coverage(metadata, df.columns.tolist())
    
    print(f"Coverage: {coverage.completeness_score:.1%}")
    if coverage.issues:
        print("\nCoverage issues:")
        for issue in coverage.issues:
            print(f"  [{issue.severity}] {issue.message}")

## 6. Save to Catalog

In [ ]:
# Save validated metadata to the catalog
if 'metadata' in dir() and 'validation' in dir():
    if validation.is_valid:
        path = assistant.save_to_catalog(metadata)
        print(f"Saved to catalog: {path}")
    else:
        print("Metadata has validation errors. Fix them before saving.")
        print("Errors:")
        for error in validation.errors:
            print(f"  - {error.message}")
else:
    print("Run the validation cell first (Section 5)")

## 7. Load from Catalog

In [ ]:
# List what's in the catalog
catalog = CatalogClient()

try:
    catalog_agencies = catalog.list_agencies()
    print("Agencies in catalog:", catalog_agencies)
    
    for agency in catalog_agencies:
        assets = catalog.list_assets(agency)
        for asset in assets:
            tables = catalog.list_tables(agency, asset)
            print(f"  {agency}/{asset}: {tables}")
except Exception as e:
    print(f"No catalog data yet or error: {e}")

In [ ]:
# Load metadata from catalog
if 'metadata' in dir():
    try:
        loaded = assistant.load_from_catalog(
            metadata.agency,
            metadata.asset,
            metadata.table_id
        )
        print(f"Loaded: {loaded.display_name}")
        print(f"Curated at: {loaded.curated_at}")
        print(f"Curated by: {loaded.curated_by}")
    except Exception as e:
        print(f"Not in catalog: {e}")

## 8. Infer Relationships (Multiple Tables)

In [ ]:
# Generate metadata for multiple tables
if 'doc' in dir() and 'tables' in dir() and len(tables) >= 2:
    all_metadata = []
    for i, t in enumerate(tables[:3]):  # First 3 tables
        print(f"Generating metadata for table {i}...")
        m = assistant.suggest_table_metadata(doc, table_index=i)
        all_metadata.append(m)
        print(f"  - {m.display_name}")
    
    print(f"\nGenerated metadata for {len(all_metadata)} tables")
else:
    print("Relationship inference requires 2+ tables in the document.")
    print(f"Current document has {len(tables) if 'tables' in dir() else 0} table(s).")

In [ ]:
# Find potential relationships (heuristic-based, no LLM)
if 'all_metadata' in dir() and len(all_metadata) >= 2:
    potential = assistant.find_potential_joins(all_metadata)
    
    print("Columns appearing in multiple tables:")
    if potential['common_columns']:
        for col, table_ids in potential['common_columns'].items():
            print(f"  {col}: {table_ids}")
    else:
        print("  (none found)")
    
    print("\nSemantic matches:")
    if potential['semantic_matches']:
        for match in potential['semantic_matches']:
            print(f"  {match[0]}.{match[1]} <-> {match[2]}.{match[3]}")
    else:
        print("  (none found)")
else:
    print("Skipped - requires metadata for 2+ tables (see previous cell)")

In [ ]:
# Use LLM to infer relationships
if 'all_metadata' in dir() and len(all_metadata) >= 2:
    print("Inferring relationships with LLM...")
    relationships = assistant.infer_relationships(all_metadata)
    
    if relationships:
        print(f"\nFound {len(relationships)} relationships:")
        for rel in relationships:
            print(f"  {rel.source_column} -> {rel.related_table_id}.{rel.target_column}")
            print(f"    Type: {rel.relationship_type}")
            if rel.description:
                print(f"    {rel.description}")
    else:
        print("\nNo relationships inferred.")
else:
    print("Skipped - requires metadata for 2+ tables (see previous cell)")

## Summary

In this notebook we:

1. Loaded a document and previewed tables
2. Used the MetadataAssistant to generate LLM-powered suggestions
3. Reviewed and validated the metadata
4. Saved curated metadata to the catalog
5. Discovered potential relationships between tables

The curated metadata is now available in the `metadata-catalog` bucket
for use by other applications and data discovery tools.